# PitchPulse: load the match export into Lakehouse Delta tables

**Status: not run yet.** This notebook has not been executed against a Fabric workspace.

1. Create a Lakehouse and attach it to this notebook as the default Lakehouse.
2. Upload the folder written by `matchmind-lakehouse export` (it holds `Tables/` and `_manifest.json`)
   to the Lakehouse under `Files/pitchpulse/`.
3. Run all cells. Each table becomes a Delta table named `pp_<table>`, then five views are created.

All data is synthetic (two fictional clubs). Numbers were computed by the PitchPulse stats engine.

In [ ]:
import json
from pyspark.sql import functions as F

BASE = "Files/pitchpulse"  # relative to the default Lakehouse
PREFIX = "pp_"  # Delta tables are named pp_events, pp_matches, ...
SPARK_TYPES = {"string": "string", "int64": "bigint", "float64": "double", "bool": "boolean", "timestamp": "timestamp"}

In [ ]:
def read_json(path):
    return json.loads(spark.read.text(path, wholetext=True).first()[0])


def load(table):
    """Read one exported table (Parquet, or the CSV fallback) with the types from its _schema.json."""
    doc = read_json(f"{BASE}/Tables/{table}/_schema.json")
    path = f"{BASE}/Tables/{table}/{doc['file']}"
    if doc["format"] == "parquet":
        return spark.read.parquet(path), doc
    df = spark.read.option("header", True).csv(path)
    return df.select([F.col(c["name"]).cast(SPARK_TYPES[c["type"]]).alias(c["name"]) for c in doc["columns"]]), doc


manifest = read_json(f"{BASE}/_manifest.json")
print(f"format: {manifest['format']}, matches: {len(manifest['matches'])}")

In [ ]:
for table in manifest["tables"]:
    df, doc = load(table)
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(PREFIX + table)
    rows = spark.table(PREFIX + table).count()
    assert rows == doc["row_count"], f"{table}: {rows} rows loaded, {doc['row_count']} exported"
    print(f"{PREFIX}{table:<18}{rows:>9} rows")

## Views

Plain Spark SQL over the Delta tables. Power BI can use the tables directly as a star schema.

In [ ]:
VIEWS = {
    "vw_match_summary": """
SELECT m.match_id, m.kickoff_utc, h.name AS home, a.name AS away, m.score_home, m.score_away,
       m.xg_home, m.xg_away, m.events, m.moments
FROM pp_matches m
JOIN pp_teams h ON h.team_id = m.home_team_id
JOIN pp_teams a ON a.team_id = m.away_team_id
""",
    "vw_xg_timeline": """
SELECT match_id, minute,
       max_by(xg_home, seq) AS xg_home, max_by(xg_away, seq) AS xg_away,
       max_by(score_home, seq) AS score_home, max_by(score_away, seq) AS score_away
FROM pp_events
GROUP BY match_id, minute
""",
    "vw_pressure_5min": """
SELECT match_id, five_min_bucket,
       round(avg(pressure_home), 1) AS avg_pressure_home, round(avg(pressure_away), 1) AS avg_pressure_away,
       max(pressure_home) AS peak_pressure_home, max(pressure_away) AS peak_pressure_away
FROM pp_events
GROUP BY match_id, five_min_bucket
""",
    "vw_better_options": """
SELECT b.match_id, b.match_clock, b.trigger, t.name AS team, p.name AS player, q.name AS teammate,
       b.shot_xg, b.alt_xg, b.alt_value, b.times_better, b.xg_gain, b.projected_xg, b.one_on_one
FROM pp_better_options b
JOIN pp_teams t ON t.team_id = b.team_id
JOIN pp_players p ON p.player_id = b.player_id
JOIN pp_players q ON q.player_id = b.teammate_id
""",
    "vw_player_leaderboard": """
SELECT p.player_id, p.name, p.team_id, count(*) AS matches,
       sum(pm.goals) AS goals, sum(pm.assists) AS assists, round(sum(pm.xg), 2) AS xg, round(sum(pm.xa), 2) AS xa,
       sum(pm.sprints) AS sprints, max(pm.top_speed_kmh) AS top_speed_kmh,
       sum(pm.better_option_shots) AS better_option_shots
FROM pp_player_match pm
JOIN pp_players p ON p.player_id = pm.player_id
GROUP BY p.player_id, p.name, p.team_id
""",
}

for name, sql in VIEWS.items():
    spark.sql(f"CREATE OR REPLACE VIEW pp_{name} AS {sql}")
    print("view", name)

In [ ]:
spark.sql("SELECT * FROM pp_vw_match_summary ORDER BY match_id").show(truncate=False)
spark.sql("SELECT * FROM pp_vw_player_leaderboard ORDER BY xg DESC LIMIT 5").show(truncate=False)